In [0]:
#Read the featurized data

df = spark.table('`science_home`.`ml-dielectric`.`featurized_merged`')

#Further dropping the features I don't need. 
#Dropping some features
df_mag_remove = df.drop('num_magnetic_sites', 'num_unique_magnetic_sites', 'total_magnetization','total_magnetization_normalized_formula_units','total_magnetization_normalized_vol', 'LUMO_character', 'HOMO_element', 'HOMO_character', 'LUMO_element')

#display(df_mag_remove.selectExpr("count(dielectric_constant) as non_null_dielectric_constant_count"))


In [0]:

#Notes : This is a feature selection cell where I checked the correlation of the structural features w.r.t. the target (dielectric constant)

import numpy as np
import pandas as pd

acsf_cols = [c for c in df_mag_remove.columns if c.startswith('acsf_')]
print(f"Total ACSF features: {len(acsf_cols)}")

# Convert to pandas for feature selection (only acsf + target)
df_fs = df_mag_remove.select(['dielectric_constant'] + acsf_cols).dropna(subset=['dielectric_constant']).toPandas()
print(f"Rows with non-null target: {len(df_fs)}")

# Drop columns that are all zeros or have zero variance
X = df_fs[acsf_cols].fillna(0)
nonzero_var_cols = X.columns[X.var() > 0].tolist()
print(f"ACSF features with non-zero variance: {len(nonzero_var_cols)}")
X = X[nonzero_var_cols]
y = df_fs['dielectric_constant'].values

# Pearson correlation with target (absolute value)
correlations = X.corrwith(pd.Series(y, name='dielectric_constant')).abs().sort_values(ascending=False)

# Select top-50 most correlated ACSF features
TOP_N = 50
top50_corr = set(correlations.head(TOP_N).index)
top50_mi = top50_corr  # Use same set for Cell 3 compatibility

print(f"\n=== Top {TOP_N} ACSF features by |Pearson correlation| ===")
print(correlations.head(TOP_N).to_string())
print(f"\nCorrelation range: {correlations.head(TOP_N).iloc[-1]:.6f} to {correlations.head(TOP_N).iloc[0]:.6f}")
print(f"Selected {len(top50_corr)} ACSF features for downstream use.")

Total ACSF features: 16016
Rows with non-null target: 7275
ACSF features with non-zero variance: 7842

=== Top 50 ACSF features by |Pearson correlation| ===
acsf_3761     0.115128
acsf_208      0.060118
acsf_211      0.059263
acsf_163      0.048551
acsf_199      0.047308
acsf_160      0.031524
acsf_196      0.028748
acsf_162      0.026035
acsf_131      0.022579
acsf_3760     0.021837
acsf_210      0.021777
acsf_130      0.016670
acsf_128      0.016035
acsf_29       0.015036
acsf_28       0.014778
acsf_30       0.014514
acsf_31       0.014272
acsf_198      0.013215
acsf_216      0.009553
acsf_35       0.007886
acsf_32       0.007803
acsf_34       0.007378
acsf_161      0.007339
acsf_33       0.006733
acsf_0        0.006616
acsf_3        0.006543
acsf_8        0.006471
acsf_1        0.006457
acsf_11       0.006451
acsf_9        0.006401
acsf_2        0.006385
acsf_10       0.006366
acsf_57       0.006114
acsf_72       0.006072
acsf_58       0.006054
acsf_75       0.005995
acsf_56       0

In [0]:
# Keep only the top-5 ACSF features most correlated with dielectric_constant (I selected manually -JK)
#selected_acsf = ['acsf_3761', 'acsf_208', 'acsf_211', 'acsf_163', 'acsf_199']
selected_acsf = ['acsf_3761']
print(f"Selected ACSF features: {len(selected_acsf)}")

#Non-ascf cols
non_acsf_cols = [c for c in df_mag_remove.columns if not c.startswith('acsf_')]
print(f"Non-ACSF columns retained: {len(non_acsf_cols)}")

keep_cols = non_acsf_cols + selected_acsf
df_selected = df_mag_remove.select(keep_cols)
print(f"Final DataFrame: {df_selected.count()} rows x {len(df_selected.columns)} columns")

display(df_selected)

Selected ACSF features: 1
Non-ACSF columns retained: 93
Final DataFrame: 153231 rows x 94 columns


id,formula,band_gap,cbm,density_atomic,dielectric_constant,e_fermi,energy_above_hull,energy_per_atom,equilibrium_reaction_energy_per_atom,formation_energy_per_atom,g_reuss,g_voigt,g_vrh,homogeneous_poisson,k_reuss,k_voigt,k_vrh,nelements,nsites,uncorrected_energy_per_atom,universal_anisotropy,vbm,volume,bulk_modulus,density,poisson_ratio,shear_modulus,space_group,youngs_modulus,lattice_parameters,MagpieData_mean_Electronegativity,MagpieData_avg_dev_Electronegativity,MagpieData_minimum_Electronegativity,MagpieData_maximum_Electronegativity,MagpieData_range_Electronegativity,MagpieData_mean_AtomicWeight,MagpieData_avg_dev_AtomicWeight,MagpieData_minimum_AtomicWeight,MagpieData_maximum_AtomicWeight,MagpieData_range_AtomicWeight,MagpieData_mean_MeltingT,MagpieData_avg_dev_MeltingT,MagpieData_minimum_MeltingT,MagpieData_maximum_MeltingT,MagpieData_range_MeltingT,MagpieData_mean_Polarizability,MagpieData_avg_dev_Polarizability,MagpieData_minimum_Polarizability,MagpieData_maximum_Polarizability,MagpieData_range_Polarizability,MagpieData_mean_GSbandgap,MagpieData_avg_dev_GSbandgap,MagpieData_minimum_GSbandgap,MagpieData_maximum_GSbandgap,MagpieData_range_GSbandgap,MagpieData_mean_NdValence,MagpieData_avg_dev_NdValence,MagpieData_minimum_NdValence,MagpieData_maximum_NdValence,MagpieData_range_NdValence,MagpieData_mean_NsValence,MagpieData_avg_dev_NsValence,MagpieData_minimum_NsValence,MagpieData_maximum_NsValence,MagpieData_range_NsValence,MagpieData_mean_NpValence,MagpieData_avg_dev_NpValence,MagpieData_minimum_NpValence,MagpieData_maximum_NpValence,MagpieData_range_NpValence,MagpieData_mean_NUnfilled,MagpieData_avg_dev_NUnfilled,MagpieData_minimum_NUnfilled,MagpieData_maximum_NUnfilled,MagpieData_range_NUnfilled,MagpieData_mean_CovalentRadius,MagpieData_avg_dev_CovalentRadius,MagpieData_minimum_CovalentRadius,MagpieData_maximum_CovalentRadius,MagpieData_range_CovalentRadius,avg_s_valence_electrons,avg_p_valence_electrons,avg_d_valence_electrons,avg_f_valence_electrons,frac_s_valence_electrons,frac_p_valence_electrons,frac_d_valence_electrons,frac_f_valence_electrons,band_center,HOMO_energy,LUMO_energy,gap_AO,acsf_3761
mp-1215927,YEr(SnPd2)2,0.0,null,19.54969477294022,null,6.44278345,0.002594258750004741,-5.85191428625,null,-0.8749791287499988,null,null,26.495397982052776,null,null,null,110.24444856859978,4,8,-5.85191428625,null,null,156.39755818352177,110.24444856859978,9.760223968368965,0.4137940927045288,20.16642189025879,166,57.02233627887035,8.289,1.8975,0.3337500000000001,1.22,2.2,0.9800000000000002,114.90810625,14.988670312499996,88.90585,167.259,78.35314999999999,1486.42,490.66999999999996,505.08,1828.05,1322.97,9.84,6.43,4.8,22.7,17.9,0.0,0.0,0.0,0.0,0.0,7.625,3.5625,0.0,10.0,10.0,1.0,1.0,0.0,2.0,2.0,0.5,0.75,0.0,2.0,2.0,2.375,2.46875,0.0,9.0,9.0,151.625,18.9375,139.0,190.0,51.0,1.0,0.5,7.625,1.5,0.09411764705882353,0.047058823529411764,0.7176470588235294,0.1411764705882353,4.071993459712238,-0.14445,-0.14445,0.0,0.0
mp-1215928,YCo4Ge,0.0,null,14.09461594851435,null,5.58832199,0.1007399383333345,-6.8300008583333325,null,-0.219036485833333,null,null,59.51977798356736,null,null,null,167.83456628075984,3,6,-6.8300008583333325,null,null,84.5676956910861,167.83456628075984,7.800811882708803,0.3847793246456424,41.894065856933594,65,116.02807244804913,3.999,1.7916666666666667,0.1905555555555554,1.22,2.01,0.7899999999999998,66.213105,9.706546666666668,58.933195,88.90585,29.972655000000003,1680.3999999999999,156.3333333333334,1211.4,1799.0,587.5999999999999,9.756666666666668,4.314444444444445,5.84,22.7,16.86,0.06383333333333334,0.10638888888888891,0.0,0.383,0.383,6.5,1.8333333333333333,1.0,10.0,9.0,2.0,0.0,2.0,2.0,0.0,0.3333333333333333,0.5555555555555555,0.0,2.0,2.0,4.166666666666667,1.6111111111111114,3.0,9.0,6.0,135.66666666666666,18.111111111111104,120.0,190.0,70.0,2.0,0.3333333333333333,6.5,0.0,0.22641509433962262,0.03773584905660377,0.7358490566037735,0.0,4.129138246368096,-0.204497,-0.204497,0.0,0.0
mp-1215929,YMgAl4,0.0,null,1

In [0]:
# Step 1: Filter the rows with non-null dielectric_constant
df_target = df_selected.dropna(subset=['dielectric_constant'])

print(f"Rows before: {df_selected.count()}")
print(f"Rows after filtering non-null target: {df_target.count()}")

#display(df_target.describe())

Rows before: 153231
Rows after filtering non-null target: 7275


In [0]:
# Step 2: Remove outliers BEFORE train-test split
from pyspark.sql.functions import col, min as spark_min, max as spark_max, mean as spark_mean, stddev

# Show distribution stats before filtering
stats = df_target.select(
    spark_min('dielectric_constant').alias('min_val'),
    spark_max('dielectric_constant').alias('max_val'),
    spark_mean('dielectric_constant').alias('mean_val'),
    stddev('dielectric_constant').alias('std_val')
).collect()[0]

print(f"Dielectric Constant Statistics (before filtering):")
print(f"  Min:  {stats['min_val']:.4f}")
print(f"  Max:  {stats['max_val']:.4f}")
print(f"  Mean: {stats['mean_val']:.4f}")
print(f"  Std:  {stats['std_val']:.4f}")
print(f"  Total rows: {df_target.count()}")

# Remove outliers: keep only dielectric_constant <= 30
df_clean = df_target.filter(col('dielectric_constant') <= 30)

stats_after = df_clean.select(
    spark_min('dielectric_constant').alias('min_val'),
    spark_max('dielectric_constant').alias('max_val'),
    spark_mean('dielectric_constant').alias('mean_val'),
    stddev('dielectric_constant').alias('std_val')
).collect()[0]

print(f"\nAfter filtering (dielectric_constant <= 30):")
print(f"  Min:  {stats_after['min_val']:.4f}")
print(f"  Max:  {stats_after['max_val']:.4f}")
print(f"  Mean: {stats_after['mean_val']:.4f}")
print(f"  Std:  {stats_after['std_val']:.4f}")
print(f"  Rows retained: {df_clean.count()} / {df_target.count()}")

Dielectric Constant Statistics (before filtering):
  Min:  1.1552
  Max:  126575.3168
  Mean: 50.9787
  Std:  1656.8128
  Total rows: 7275

After filtering (dielectric_constant <= 30):
  Min:  1.1552
  Max:  29.9928
  Mean: 12.0750
  Std:  6.1026
  Rows retained: 6396 / 7275


In [0]:
# Log-transform the target to handle right-skew and multiplicative scaling
# Since min = 1.15, np.log() is safe (no log(0) risk)

import numpy as np
from pyspark.sql.functions import log as spark_log, col

df_clean = df_clean.withColumn('dielectric_constant', spark_log(col('dielectric_constant')))

# Verify transformed distribution
from pyspark.sql.functions import min as spark_min, max as spark_max, mean as spark_mean, stddev, skewness

stats_log = df_clean.select(
    spark_min('dielectric_constant').alias('min_val'),
    spark_max('dielectric_constant').alias('max_val'),
    spark_mean('dielectric_constant').alias('mean_val'),
    stddev('dielectric_constant').alias('std_val'),
    skewness('dielectric_constant').alias('skewness')
).collect()[0]

print("Dielectric Constant after log-transform:")
print(f"  Min:      {stats_log['min_val']:.4f}")
print(f"  Max:      {stats_log['max_val']:.4f}")
print(f"  Mean:     {stats_log['mean_val']:.4f}")
print(f"  Std:      {stats_log['std_val']:.4f}")
print(f"  Skewness: {stats_log['skewness']:.4f}")
print(f"\nNote: At prediction time, apply np.exp(prediction) to get actual dielectric constant.")

Dielectric Constant after log-transform:
  Min:      0.1443
  Max:      3.4010
  Mean:     2.3623
  Std:      0.5211
  Skewness: -0.2856

Note: At prediction time, apply np.exp(prediction) to get actual dielectric constant.


In [0]:
# Impute missing numeric values with 0 for feature columns only
# (dielectric_constant is already non-null and log-transformed from Cell 6)
from pyspark.sql.functions import col, count, when

numeric_cols = [c for c in df_clean.columns if c not in ['id', 'formula', 'dielectric_constant'] and df_clean.schema[c].dataType.typeName() in ('double', 'float', 'integer')]

df_imputed = df_clean.fillna(0, subset=numeric_cols)

remaining_nulls = df_imputed.select([count(when(col(c).isNull(), c)).alias(c) for c in numeric_cols]).toPandas().sum().sum()
print(f"Feature columns imputed: {len(numeric_cols)}")
print(f"Remaining nulls in feature columns: {int(remaining_nulls)}")

Feature columns imputed: 91
Remaining nulls in feature columns: 0


In [0]:
# Step 4: Feature scaling using sklearn (pandas-based, faster for this data size)
# Note: dielectric_constant is already log-transformed (Cell 6) - not scaled here
from sklearn.preprocessing import StandardScaler
import pandas as pd

df_pd = df_imputed.toPandas()

# StandardScaler is applied only to feature columns, not the log-transformed target
feature_cols = [c for c in df_pd.select_dtypes(include=['number']).columns if c not in ['id', 'formula', 'dielectric_constant']]

scaler = StandardScaler()
df_pd[feature_cols] = scaler.fit_transform(df_pd[feature_cols])

print(f"Feature columns scaled: {len(feature_cols)}")
print(f"DataFrame shape: {df_pd.shape}")
print(f"Target (log-transformed) sample: {df_pd['dielectric_constant'].head().tolist()}")
df_pd[['id', 'formula', 'dielectric_constant'] + feature_cols[:5]].head()

Feature columns scaled: 91
DataFrame shape: (6396, 94)
Target (log-transformed) sample: [2.7160922679327397, 3.1635956508310175, 3.282348126545767, 2.9392481079411668, 2.7763762430188454]


,id,formula,dielectric_constant,band_gap,cbm,density_atomic,e_fermi,energy_above_hull
0,mp-1216058,Y2TeS2,2.716092,-0.350146,0.211031,1.318216,0.393320,-0.186404
1,mp-1216121,Y2FeSbO7,3.163596,-0.384742,1.258283,-0.718427,1.305284,-0.186404
2,mp-1216330,VFeSbO6,3.282348,-1.328766,0.403328,-0.892362,1.294378,-0.146752
3,mp-1216355,VCrSbO6,2.939248,-1.447555,-1.844322,-0.906883,1.793276,-0.186404
4,mp-1216423,VAg3HgO4,2.776376,-0.588726,-0.640563,-0.296051,-0.153931,-0.186404


In [0]:
# Step 5: Stratified Train/Test split (bin log-transformed target into quantiles for stratification)
from sklearn.model_selection import train_test_split
import numpy as np

# Bin log(dielectric_constant) into quantile groups for stratified splitting
df_pd['target_bin'] = pd.qcut(df_pd['dielectric_constant'], q=10, labels=False, duplicates='drop')

train_df, test_df = train_test_split(
    df_pd, test_size=0.2, random_state=42, stratify=df_pd['target_bin']
)

# Drop the temporary binning column
train_df = train_df.drop(columns=['target_bin'])
test_df = test_df.drop(columns=['target_bin'])

# Verify distributions match (values are in log-space)
print(f"train_data: {len(train_df)} rows")
print(f"test_data: {len(test_df)} rows")
print(f"\nTarget distribution (log-space):")
print(f"  Train mean: {train_df['dielectric_constant'].mean():.4f}, median: {train_df['dielectric_constant'].median():.4f}")
print(f"  Test  mean: {test_df['dielectric_constant'].mean():.4f}, median: {test_df['dielectric_constant'].median():.4f}")
print(f"\nTarget distribution (original scale via exp):")
print(f"  Train mean: {np.exp(train_df['dielectric_constant']).mean():.4f}, median: {np.exp(train_df['dielectric_constant']).median():.4f}")
print(f"  Test  mean: {np.exp(test_df['dielectric_constant']).mean():.4f}, median: {np.exp(test_df['dielectric_constant']).median():.4f}")
print("\nSaved to UC. Note: dielectric_constant is log-transformed. Apply np.exp() to predictions.")

#Saving test and training data in UC
spark.createDataFrame(train_df).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('science_home.`ml-dielectric`.train_data')
spark.createDataFrame(test_df).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('science_home.`ml-dielectric`.test_data')


train_data: 5116 rows
test_data: 1280 rows

Target distribution (log-space):
  Train mean: 2.3624, median: 2.3601
  Test  mean: 2.3616, median: 2.3601

Target distribution (original scale via exp):
  Train mean: 12.0736, median: 10.5919
  Test  mean: 12.0807, median: 10.5916

Saved to UC. Note: dielectric_constant is log-transformed. Apply np.exp() to predictions.


In [0]:
# Save prediction data (null dielectric_constant rows) to UC
# These rows have unknown dielectric_constant - model predictions will be in log-space
# Apply np.exp(prediction) to get actual dielectric constant values
from pyspark.sql.functions import col

df_prediction = df_selected.filter(col('dielectric_constant').isNull())
df_pred_pd = df_prediction.fillna(0, subset=numeric_cols).toPandas()
df_pred_pd[feature_cols] = scaler.transform(df_pred_pd[feature_cols])

spark.createDataFrame(df_pred_pd).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('science_home.`ml-dielectric`.prediction_data')
print(f"prediction_data: {len(df_pred_pd)} rows")
print(f"Features scaled using same scaler as train/test.")
print(f"\nReminder: Model predictions will be in log-space.")
print(f"Use np.exp(y_pred) to convert back to actual dielectric constant.")

prediction_data: 145956 rows
Features scaled using same scaler as train/test.

Reminder: Model predictions will be in log-space.
Use np.exp(y_pred) to convert back to actual dielectric constant.
